# Conector Otimizado DB2 para Spark

Módulo corporativo para conexão de alta performance com o **IBM DB2** no Apache Spark.

---

### 📌 Guia Operacional: Boas Práticas e Limitações do DB2 via Spark JDBC

1. **Não usar CTE (`WITH`):**
   - O Spark encapsula a consulta JDBC em uma subconsulta no formato `(SELECT ...) T`. O DB2 rejeita subconsultas envolvendo `WITH` nesse formato. Construa subconsultas simples ou realize a lógica diretamente no Spark.
2. **Consultas Simples na Origem:**
   - Prefira `SELECT ... FROM SCHEMA.TABELA WHERE ...`. Joins pesados, agregações analíticas e transformações complexas devem ser processados no Spark distribuído.
3. **Referenciar Sempre com Schema Completo:**
   - Sempre utilize a notação `SCHEMA.TABELA` (ex: `DB2GFP.TRAN_RLZD_INST_PCT`). Omitir o schema faz o DB2 assumir o usuário de conexão como schema padrão (`SQLCODE -204`).
4. **Espaço Obrigatório em Listas `IN`:**
   - Em filtros `IN`, use sempre vírgula seguida de espaço: `IN (1, 6, 32)`. Em configurações regionais onde a vírgula é separador decimal, `IN (1,6)` pode ser interpretado pelo DB2 como o número decimal `1.6`.
5. **Representar Datas Explicitamente:**
   - Utilize a função explícita `DATE('YYYY-MM-DD')` (ex: `DATE('2026-07-01')`) para evitar ambiguidades com formatos regionais de data do DB2.
6. **Pushdown Estrito de Filtros:**
   - Aplique todos os filtros de data e identificadores diretamente na cláusula `WHERE` da leitura JDBC para reduzir a volumetria trafegada pela rede.
7. **Validação Prévia de Parâmetros:**
   - Valide tipos de dados (códigos numéricos inteiros, formatos ISO de data) antes de montar as strings de consulta.
8. **Sanitização e Escape de Textos:**
   - Trate strings de texto duplicando aspas simples (`''`) para prevenir quebras de sintaxe SQL.
9. **Evitar Aliases Desnecessários no DB2:**
   - Evite renomeações ou aliases complexos na consulta SQL do DB2 quando uma seleção direta de colunas for suficiente.
10. **Leitura Única por Tabela:**
    - Leia cada tabela apenas uma única vez na rotina e reutilize o DataFrame resultante com cache/persist no Spark.
11. **Recorte de Domínio Baseado no Público:**
    - Ao consultar tabelas de domínio, filtre exclusivamente pelos códigos encontrados no público principal já carregado no Spark.
12. **Lista Vazia $	o$ `WHERE 1 = 0`:**
    - Quando não houver códigos para filtrar em um domínio, utilize uma condição segura como `WHERE 1 = 0` em vez de gerar `IN ()` (que gera erro de sintaxe SQL no DB2).
13. **Diagnóstico Preciso de Falhas:**
    - Em caso de erro, inspecione a mensagem original da JVM buscando `SQLCODE`, `SQLSTATE` e `SQLERRMC` para descobrir a causa real (ex: lock, timeout, tabela inexistente, estouro de log).

---

### Recursos de Destaque:
- **Zero Locks / Isolamento Não Bloqueante:** Suporte a `isolationLevel = READ_UNCOMMITTED` (DB2 `UR`) para evitar travamentos em tabelas transacionais de produção.
- **Catálogo de Metadados Estrito em `SYSIBM`:** Consultas de tabelas, colunas e índices utilizam **unicamente** o schema `SYSIBM` (`SYSIBM.SYSTABLES`, `SYSIBM.SYSCOLUMNS`, `SYSIBM.SYSINDEXES`, `SYSIBM.SYSKEYS`).
- **Engenharia de Dados:** `sql`, `table` e leitura particionada JDBC paralela por chunks.
- **Ciência de Dados:** `describe`, `limit`, `list_columns`.


## 1. Conector IBM DB2 (`ConectorDb2Spark`)


In [ ]:
%%spark

class ConectorDb2Spark:
    """
    Conector corporativo de alta performance para IBM DB2 no Apache Spark.
    """
    DRIVER_PADRAO = "com.ibm.db2.jcc.DB2Driver"

    def __init__(
        self,
        spark: SparkSession,
        env: Optional[Dict[str, str]] = None,
        isolamento_padrao: str = "READ_UNCOMMITTED",
    ) -> None:
        self.spark = spark
        self.env = env or dict(os.environ)
        self.isolamento_padrao = isolamento_padrao

        def extrair_obrigatorio(chave: str) -> str:
            val = self.env.get(chave)
            if val is None or not str(val).strip():
                raise ValueError(f"Variável obrigatória do DB2 não configurada: '{chave}'")
            return str(val).strip()

        def extrair_opcional(chave: str) -> Optional[str]:
            val = self.env.get(chave)
            return str(val).strip() if val and str(val).strip() else None

        self.usuario = extrair_obrigatorio("DB2_USER")
        self.senha = extrair_obrigatorio("DB2_PASSWORD")
        self.host = extrair_obrigatorio("DB2_HOST")
        self.porta = extrair_obrigatorio("DB2_PORTA")
        self.database = extrair_obrigatorio("DB2_DATABASE")
        self.driver = extrair_opcional("DB2_DRIVER") or self.DRIVER_PADRAO

        if self.driver != self.DRIVER_PADRAO:
            raise ValueError(f"Driver DB2 inválido: {self.driver}")

        self.url = f"jdbc:db2://{self.host}:{self.porta}/{self.database}"

    @staticmethod
    def _normalizar_identificador(valor: str, nome_parametro: str) -> str:
        res = (valor or "").strip().upper()
        if not res:
            raise ValueError(f"Parâmetro '{nome_parametro}' não pode ser vazio.")
        if not re.fullmatch(r"[A-Z][A-Z0-9_#$@]{0,127}", res):
            raise ValueError(
                f"Identificador DB2 inválido para '{nome_parametro}': {valor!r}. "
                "Utilize nomes válidos de objeto sem caracteres especiais ou pontos."
            )
        return res

    @staticmethod
    def _normalizar_limite(valor: Any) -> str:
        if valor is None:
            raise ValueError("Valor de limite de partição não pode ser nulo.")
        if isinstance(valor, (datetime.date, datetime.datetime)):
            return valor.strftime("%Y-%m-%d")
        if isinstance(valor, bool):
            raise TypeError("Valor booleano inválido como limite de partição.")
        return str(valor).strip()

    @staticmethod
    def _converter_limite_comparavel(valor: str) -> Union[decimal.Decimal, datetime.date, str]:
        texto = valor.strip()
        try:
            return decimal.Decimal(texto)
        except decimal.InvalidOperation:
            pass
        if re.fullmatch(r"\d{4}-\d{2}-\d{2}", texto):
            return datetime.date.fromisoformat(texto)
        return texto

    @staticmethod
    def _estimar_tamanho_coluna(tipo: str, tamanho: Any) -> int:
        try:
            tam_final = max(0, int(tamanho or 0))
        except (TypeError, ValueError):
            tam_final = 0

        tamanhos_fixos = {
            "SMALLINT": 2,
            "INTEGER": 4,
            "BIGINT": 8,
            "REAL": 4,
            "FLOAT": 8,
            "DOUBLE": 8,
            "DATE": 4,
            "TIME": 3,
            "TIMESTAMP": 12,
            "TIMESTMP": 12,
        }
        if tipo in tamanhos_fixos:
            return tamanhos_fixos[tipo]
        if tipo in {"GRAPHIC", "VARGRAPHIC", "LONGVARG"}:
            return tam_final * 2
        return tam_final or 32

    def _obter_configuracao_spark_int(self, chave: str) -> Optional[int]:
        try:
            val = self.spark.sparkContext.getConf().get(chave)
            return int(val) if val is not None else None
        except (TypeError, ValueError, AttributeError):
            return None

    def sql(
        self,
        query: str,
        fetchsize: Optional[int] = 10_000,
        query_timeout: Optional[int] = 900,
        isolation_level: Optional[str] = None,
        partition_column: Optional[str] = None,
        lower_bound: Any = None,
        upper_bound: Any = None,
        num_partitions: Optional[int] = None,
        show: bool = False,
        truncate: bool = True,
        n: int = 20,
    ) -> DataFrame:
        """
        Executa uma consulta SQL no DB2 e retorna um DataFrame Spark lazy.
        """
        instrucao_sql = (query or "").strip()
        if not instrucao_sql:
            raise ValueError("A instrução SQL não pode ser vazia.")
        if instrucao_sql.endswith(";"):
            instrucao_sql = instrucao_sql[:-1].strip()

        reader = (
            self.spark.read
            .format("jdbc")
            .option("url", self.url)
            .option("driver", self.driver)
            .option("user", self.usuario)
            .option("password", self.senha)
            .option("dbtable", f"({instrucao_sql}) T")
        )

        if fetchsize is not None:
            if int(fetchsize) <= 0:
                raise ValueError("fetchsize deve ser maior que zero.")
            reader = reader.option("fetchsize", int(fetchsize))

        if query_timeout is not None:
            if int(query_timeout) <= 0:
                raise ValueError("query_timeout deve ser maior que zero.")
            reader = reader.option("queryTimeout", int(query_timeout))

        isolamento = isolation_level or self.isolamento_padrao
        if isolamento:
            reader = reader.option("isolationLevel", isolamento)

        parametros_particao = [partition_column, lower_bound, upper_bound, num_partitions]
        if any(v is not None for v in parametros_particao):
            if any(v is None for v in parametros_particao):
                raise ValueError(
                    "Para leitura particionada paralela no DB2, informe todos os 4 parâmetros: "
                    "partition_column, lower_bound, upper_bound e num_partitions."
                )

            coluna_part_norm = self._normalizar_identificador(partition_column, "partition_column")
            lb_norm = self._normalizar_limite(lower_bound)
            ub_norm = self._normalizar_limite(upper_bound)
            num_parts_norm = int(num_partitions)

            if self._converter_limite_comparavel(lb_norm) >= self._converter_limite_comparavel(ub_norm):
                raise ValueError("lower_bound deve ser estritamente menor que upper_bound.")
            if num_parts_norm <= 0:
                raise ValueError("num_partitions deve ser maior que zero.")

            reader = (
                reader
                .option("partitionColumn", coluna_part_norm)
                .option("lowerBound", lb_norm)
                .option("upperBound", ub_norm)
                .option("numPartitions", num_parts_norm)
            )

        df = reader.load()
        if show:
            df.show(n=int(n), truncate=truncate)
        return df

    def table(
        self,
        schema: str,
        table_name: str,
        filter_cols: Optional[List[str]] = None,
        date_col: Optional[str] = None,
        filter_date: Any = None,
        partition_column: Optional[str] = None,
        lower_bound: Any = None,
        upper_bound: Any = None,
        fetchsize: Optional[int] = 10_000,
        num_partitions: Optional[int] = None,
        show: bool = False,
        truncate: bool = True,
        n: int = 20,
    ) -> DataFrame:
        """
        Gera e executa uma consulta otimizada direta sobre uma tabela do DB2.
        """
        schema_final = self._normalizar_identificador(schema, "schema")
        tabela_final = self._normalizar_identificador(table_name, "table_name")

        colunas_retorno = []
        if filter_cols:
            if isinstance(filter_cols, str):
                raise TypeError("filter_cols deve ser uma lista de strings com os nomes das colunas.")
            for col in filter_cols:
                c_norm = self._normalizar_identificador(col, "filter_cols")
                if c_norm not in colunas_retorno:
                    colunas_retorno.append(c_norm)

        col_particao_final = self._normalizar_identificador(partition_column, "partition_column") if partition_column else None
        colunas_leitura = list(colunas_retorno)
        remover_col_particao_apos = False

        if colunas_leitura and col_particao_final and col_particao_final not in colunas_leitura:
            colunas_leitura.append(col_particao_final)
            remover_col_particao_apos = True

        selecao_sql = ", ".join(colunas_leitura) if colunas_leitura else "*"

        clausulas_where = []
        if date_col and filter_date is not None:
            col_data_final = self._normalizar_identificador(date_col, "date_col")
            if isinstance(filter_date, (list, tuple)):
                if len(filter_date) == 1:
                    dt_ini = self._normalizar_limite(filter_date[0])
                    clausulas_where.append(f"{col_data_final} >= '{dt_ini}'")
                elif len(filter_date) >= 2:
                    dt_ini = self._normalizar_limite(filter_date[0])
                    dt_fim = self._normalizar_limite(filter_date[1])
                    clausulas_where.append(f"{col_data_final} >= '{dt_ini}' AND {col_data_final} < '{dt_fim}'")
            else:
                dt_ini = self._normalizar_limite(filter_date)
                clausulas_where.append(f"{col_data_final} >= '{dt_ini}'")

        filtro_where = (" WHERE " + " AND ".join(clausulas_where)) if clausulas_where else ""
        sql_gerada = f"SELECT {selecao_sql} FROM {schema_final}.{tabela_final}{filtro_where}"

        df = self.sql(
            query=sql_gerada,
            fetchsize=fetchsize,
            partition_column=col_particao_final,
            lower_bound=lower_bound,
            upper_bound=upper_bound,
            num_partitions=num_partitions,
            show=False,
        )

        if remover_col_particao_apos and col_particao_final in df.columns:
            df = df.drop(col_particao_final)

        if show:
            df.show(n=int(n), truncate=truncate)
        return df

    def describe(
        self,
        schema: str,
        table_name: str,
        show: bool = True,
        truncate: bool = False,
    ) -> Dict[str, Any]:
        """
        Diagnostica uma tabela DB2 consultando EXCLUSIVAMENTE o catálogo SYSIBM.
        Retorna sugestões de parâmetros JDBC otimizados para Engenharia e Ciência de Dados.
        """
        schema_final = self._normalizar_identificador(schema, "schema")
        tabela_final = self._normalizar_identificador(table_name, "table_name")

        # 1. Metadados da Tabela em SYSIBM.SYSTABLES
        sql_tabela = f"""
            SELECT
                TYPE AS TIPO_OBJETO,
                CARDF AS ESTIMATIVA_LINHAS
            FROM SYSIBM.SYSTABLES
            WHERE CREATOR = '{schema_final}'
              AND NAME = '{tabela_final}'
        """
        metadados_tabela = self.sql(sql_tabela).collect()
        if not metadados_tabela:
            raise ValueError(f"Tabela ou View não encontrada no catálogo SYSIBM: '{schema_final}.{tabela_final}'")

        try:
            estimativa_linhas = float(metadados_tabela[0]["ESTIMATIVA_LINHAS"] or -1)
        except (TypeError, ValueError):
            estimativa_linhas = -1

        # 2. Metadados das Colunas em SYSIBM.SYSCOLUMNS
        sql_colunas = f"""
            SELECT
                NAME AS COLUNA,
                COLNO AS POSICAO,
                COLTYPE AS TIPO_DB2,
                LENGTH AS TAMANHO,
                NULLS AS ACEITA_NULO
            FROM SYSIBM.SYSCOLUMNS
            WHERE TBCREATOR = '{schema_final}'
              AND TBNAME = '{tabela_final}'
            ORDER BY COLNO
        """
        colunas = self.sql(sql_colunas).collect()
        if not colunas:
            raise ValueError(f"Nenhuma coluna localizada em SYSIBM.SYSCOLUMNS para '{schema_final}.{tabela_final}'.")

        # 3. Metadados de Índices em SYSIBM.SYSINDEXES e SYSIBM.SYSKEYS
        sql_indices = f"""
            SELECT
                K.COLNAME AS COLUNA,
                K.COLSEQ AS POSICAO_INDICE
            FROM SYSIBM.SYSINDEXES I
            INNER JOIN SYSIBM.SYSKEYS K
                ON I.CREATOR = K.IXCREATOR
               AND I.NAME = K.IXNAME
            WHERE I.TBCREATOR = '{schema_final}'
              AND I.TBNAME = '{tabela_final}'
            ORDER BY K.COLNAME, K.COLSEQ
        """
        indices = self.sql(sql_indices).collect()
        colunas_indice_lider = set()
        for idx in indices:
            try:
                if int(idx["POSICAO_INDICE"]) == 1:
                    colunas_indice_lider.add(str(idx["COLUNA"]).strip().upper())
            except (TypeError, ValueError):
                continue

        tipos_data = {"DATE", "TIMESTAMP", "TIMESTMP"}
        tipos_numericos = {"SMALLINT", "INTEGER", "BIGINT", "DECIMAL", "NUMERIC", "DECFLOAT", "REAL", "FLOAT", "DOUBLE"}
        tipos_lob = {"BLOB", "CLOB", "DBCLOB", "XML", "LONGVAR", "LONGVARG", "LONGVARB"}
        tipos_particao = tipos_data | tipos_numericos

        filtrar_cols = []
        col_data = []
        col_particao = []
        tipo_por_coluna = {}
        tamanho_linha_estimado = 0
        possui_lob = False

        for col in colunas:
            nome_c = str(col["COLUNA"]).strip().upper()
            tipo_c = str(col["TIPO_DB2"] or "").strip().upper()
            nulo_c = str(col["ACEITA_NULO"] or "").strip().upper()

            filtrar_cols.append(nome_c)
            tipo_por_coluna[nome_c] = tipo_c
            tamanho_linha_estimado += self._estimar_tamanho_coluna(tipo_c, col["TAMANHO"])

            if tipo_c in tipos_lob:
                possui_lob = True
            if tipo_c in tipos_data:
                col_data.append(nome_c)
            if tipo_c in tipos_particao and nulo_c == "N" and nome_c in colunas_indice_lider:
                col_particao.append(nome_c)

        # 4. Cálculo de Bounds se houver colunas candidatas
        bounds = {}
        if col_particao:
            agregacoes = []
            for pos, nome_c in enumerate(col_particao):
                agregacoes.extend([f"MIN({nome_c}) AS LB_{pos}", f"MAX({nome_c}) AS UB_{pos}"])
            campos_bounds = ", ".join(agregacoes)
            sql_bounds = f"SELECT {campos_bounds} FROM {schema_final}.{tabela_final}"
            linha_bounds = self.sql(sql_bounds).collect()[0].asDict(recursive=True)

            for pos, nome_c in enumerate(col_particao):
                lb_val = linha_bounds.get(f"LB_{pos}")
                ub_val = linha_bounds.get(f"UB_{pos}")
                bounds[nome_c] = {
                    "lower_bound": self._normalizar_limite(lb_val) if lb_val is not None else None,
                    "upper_bound": self._normalizar_limite(ub_val) if ub_val is not None else None,
                }

        # 5. Sugestão Inteligente de fetchsize e partições
        if possui_lob or tamanho_linha_estimado > 16_384:
            fetchsize_sugerido = 1_000
        elif tamanho_linha_estimado > 4_096:
            fetchsize_sugerido = 5_000
        else:
            fetchsize_sugerido = 10_000

        try:
            capacidade_spark = max(1, int(self.spark.sparkContext.defaultParallelism))
        except Exception:
            capacidade_spark = 1

        exec_cores = self._obter_configuracao_spark_int("spark.executor.cores")
        exec_instances = self._obter_configuracao_spark_int("spark.executor.instances")
        if exec_cores and exec_instances:
            capacidade_spark = max(capacidade_spark, exec_cores * exec_instances)
        capacidade_jdbc = min(capacidade_spark, 16)

        if not col_particao:
            num_parts_sugerido = None
        elif estimativa_linhas <= 250_000:
            num_parts_sugerido = 1
        elif estimativa_linhas <= 2_000_000:
            num_parts_sugerido = min(4, capacidade_jdbc)
        elif estimativa_linhas <= 20_000_000:
            num_parts_sugerido = min(8, capacidade_jdbc)
        else:
            num_parts_sugerido = capacidade_jdbc

        if show:
            print("=" * 80)
            print(f"DIAGNÓSTICO SYSIBM DB2: {schema_final}.{tabela_final}")
            print("=" * 80)
            print(f"- Estimativa de Linhas (CARDF): {int(estimativa_linhas):,}" if estimativa_linhas >= 0 else "- Estimativa de Linhas: Desconhecida")
            print(f"- Tamanho Estimado de Linha: {tamanho_linha_estimado} bytes | Possui LOBs: {'SIM' if possui_lob else 'NÃO'}")
            print(f"- fetchsize sugerido: {fetchsize_sugerido:,}")
            print(f"- Quantidade de partições sugerida: {num_parts_sugerido or '1 (Sem coluna de partição indexada elegível)'}")
            print(f"- Colunas com índice elegíveis para partição: {col_particao or 'Nenhuma'}")
            print("=" * 80)

        return {
            "filter_cols": filtrar_cols,
            "date_col": col_data,
            "partition_column": col_particao,
            "bounds": bounds,
            "fetchsize": fetchsize_sugerido,
            "num_partitions": num_parts_sugerido,
            "estimativa_linhas": estimativa_linhas,
            "tamanho_linha_bytes": tamanho_linha_estimado,
        }

    def limit(self, schema: str, table_name: str, limit: int = 100) -> DataFrame:
        """
        Obtém uma amostra rápida de linhas do DB2 para análise exploratória.
        """
        schema_final = self._normalizar_identificador(schema, "schema")
        tabela_final = self._normalizar_identificador(table_name, "table_name")
        sql_query = f"SELECT * FROM {schema_final}.{tabela_final} FETCH FIRST {int(limit)} ROWS ONLY"
        return self.sql(sql_query, fetchsize=min(limit, 5000))

    def list_columns(self, schema: str, table_name: str) -> DataFrame:
        """
        Retorna um DataFrame Spark contendo as colunas, tipos e tamanhos via SYSIBM.SYSCOLUMNS.
        """
        schema_final = self._normalizar_identificador(schema, "schema")
        tabela_final = self._normalizar_identificador(table_name, "table_name")
        sql_query = f"""
            SELECT
                NAME AS NOME_COLUNA,
                COLNO AS POSICAO,
                COLTYPE AS TIPO_DB2,
                LENGTH AS TAMANHO_BYTES,
                NULLS AS ACEITA_NULO
            FROM SYSIBM.SYSCOLUMNS
            WHERE TBCREATOR = '{schema_final}'
              AND TBNAME = '{tabela_final}'
            ORDER BY COLNO
        """
        return self.sql(sql_query)


def criar_conector_db2_spark(env: Optional[Dict[str, str]] = None) -> ConectorDb2Spark:
    return ConectorDb2Spark(spark=spark, env=env)

print("Módulo gerenciador_db2_spark carregado com sucesso (ConectorDb2Spark).")
